# 점수 예측 (Score Prediction)

NBA play-by-play 데이터를 사용해 **각 이벤트 시점에서 두 팀이 경기 종료까지 추가로 득점할 점수**를 예측하는 LSTM 모델을 학습합니다.

전체 흐름:
1. **데이터 다운로드** — Kaggle에서 `pbp2023.csv`(2023시즌 NBA 이벤트 로그) 사용
2. **전처리** — clock 파싱, 누적/잔여 점수 계산, 이벤트 타입 정수 인코딩, 게임 단위 시퀀스화
3. **Dataset / DataLoader** — 가변 길이 시퀀스 패딩 및 valid_mask 생성
4. **모델 학습** — Embedding + LSTM + MLP regressor, masked SmoothL1 손실
5. **시각화** — 학습 곡선과 테스트셋 예측 비교

## 학습 데이터 설치


In [2]:
import os
from pathlib import Path

# Windows에서 Jupyter + PyTorch + NumPy 동시 사용 시 OpenMP 런타임 중복 로딩으로
# 커널이 죽는 문제가 있어 우회 옵션을 켠다. 다른 OS에서는 영향 없음.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd


In [3]:
# dataset/pbp2023.csv 가 없으면 kagglehub 으로 자동 다운로드한다.
# (Kaggle 계정 인증이 필요한 경우 콘솔에서 한 번 로그인해두어야 한다.)
data_dir = Path("dataset")
csv_path = data_dir / "pbp2023.csv"

if not csv_path.exists():
    try:
        import kagglehub

        downloaded_path = Path(
            kagglehub.dataset_download(
                "szymonjwiak/nba-play-by-play-data-1997-2023",
                output_dir="dataset",
            )
        )
        csv_path = downloaded_path / "pbp2023.csv"
    except Exception as exc:
        raise FileNotFoundError(
            "dataset not found"
        ) from exc

print(f"Using dataset file: {csv_path}")


FileNotFoundError: dataset not found

In [ ]:
# 원본 CSV 미리보기 — gameid, season 은 0으로 시작할 수 있어 문자열로 읽는다.
raw_df = pd.read_csv(csv_path, dtype={"gameid": str, "season": str})
print("rows:", len(raw_df), "| games:", raw_df["gameid"].nunique())
print("columns:", list(raw_df.columns))
raw_df.head(10)


: 

## 데이터 전처리

원본 play-by-play 로그는 한 행 = 한 이벤트(슛/리바운드/파울 등) 형태다. 모델 입력으로 쓰기 위해 다음 작업을 수행한다.

- ISO 8601 형식 clock(`PT11M38.00S`) → 남은 초(float) 변환
- 누락된 점수 컬럼 forward-fill → 매 이벤트의 **현재 누적 점수** 계산
- 게임별 최종 점수에서 현재 점수를 빼서 **잔여 점수(타깃)** 계산
- 점수 증가 패턴에서 게임별 home/away 팀을 추론해 **`team_side`** (홈=1.0, 어웨이=0.0, 무귀속=0.5) 컬럼 생성
- 이벤트 타입 문자열을 정수 ID로 인코딩 (LSTM 임베딩 입력용)
- 게임 단위로 정렬·검증 후 시퀀스 리스트 생성

### 전처리 함수 정의

In [ ]:
import re
from typing import Dict

import torch

: 

In [ ]:
# clock 문자열 파서 — ISO 8601 duration 일부 형식 (예: "PT11M38.00S", "PT45.5S")
_CLOCK_PATTERN = re.compile(r"^PT(?:(\d+)M)?(\d+(?:\.\d+)?)S$")

# LSTM 입력으로 사용할 수치형 피처 (이벤트 임베딩과 concat 됨)
NUMERIC_FEATURE_COLUMNS = [
    "elapsed_game_sec",       # 경기 시작 후 경과 시간 (초)
    "period",                 # 쿼터 번호 (1~4 정규, 5+ 연장)
    "clock_sec_remaining",    # 현재 쿼터에 남은 시간 (초)
    "current_home_points",    # 현 시점 홈팀 누적 점수
    "current_away_points",    # 현 시점 원정팀 누적 점수
    "team_side",              # 이 이벤트의 귀속: 홈=1.0, 어웨이=0.0, 무귀속=0.5
]

# 회귀 타깃 — 경기 종료 시점까지 두 팀이 추가로 득점할 점수
TARGET_COLUMNS = ["remaining_home_points", "remaining_away_points"]

In [ ]:
def parse_clock_to_sec(clock: str) -> float:
    """clock 문자열("PT11M38.00S")을 남은 초(float)로 변환."""
    if pd.isna(clock):
        return 0.0

    match = _CLOCK_PATTERN.match(str(clock).strip())
    if match is None:
        raise ValueError(f"Unexpected clock format: {clock}")

    minutes = float(match.group(1) or 0.0)
    seconds = float(match.group(2))
    return minutes * 60.0 + seconds

: 

In [ ]:
def normalize_event_type(type_raw: str, desc: str) -> str:
    """type 컬럼이 NaN 인 경우 desc 텍스트에서 Block / Steal 키워드를 추출해 보강."""
    type_clean = "" if pd.isna(type_raw) else " ".join(str(type_raw).split())
    if type_clean:
        return type_clean

    desc_upper = "" if pd.isna(desc) else str(desc).upper()
    if "BLOCK" in desc_upper:
        return "Block"
    if "STEAL" in desc_upper:
        return "Steal"
    return "Unknown"

: 

In [ ]:
def build_preprocessed_df(csv_path: str) -> pd.DataFrame:
    """원본 CSV 를 모델 입력 가능한 DataFrame 으로 변환.

    이 함수는 단순 변환뿐 아니라 일관성 검증(assertion)도 함께 수행한다.
    검증이 실패하면 학습 단계에서 잘못된 신호를 학습하므로 즉시 ValueError 를 낸다.
    """
    df = pd.read_csv(csv_path, dtype={"gameid": str, "season": str})

    # 필수 컬럼 존재 확인 — team_side 추론에 team 컬럼이 필요하다
    required_cols = {
        "gameid",
        "period",
        "clock",
        "h_pts",
        "a_pts",
        "team",
        "type",
        "desc",
    }
    missing_cols = sorted(required_cols - set(df.columns))
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # 같은 (gameid, period, clock) 을 가진 행의 원래 순서를 보존하기 위한 보조 키
    df["_row_id"] = np.arange(len(df), dtype=np.int64)

    df["period"] = pd.to_numeric(df["period"], errors="raise").astype(np.int64)
    df["clock_sec_remaining"] = df["clock"].apply(parse_clock_to_sec).astype(np.float32)

    # 정렬 순서가 매우 중요하다.
    #   - period 오름차순 (1Q → 4Q → OT)
    #   - clock_sec_remaining 내림차순 (각 쿼터 내에서는 시계가 줄어드니 큰 값이 먼저)
    #   - _row_id 오름차순 (동시 이벤트는 원본 순서 유지)
    df = df.sort_values(
        ["gameid", "period", "clock_sec_remaining", "_row_id"],
        ascending=[True, True, False, True],
    ).reset_index(drop=True)

    df["h_pts"] = pd.to_numeric(df["h_pts"], errors="coerce")
    df["a_pts"] = pd.to_numeric(df["a_pts"], errors="coerce")

    # 점수 컬럼은 득점 이벤트가 발생한 행에만 값이 있고 나머지는 NaN.
    # 게임별로 forward-fill 하면 모든 이벤트에서 "그 시점까지의 누적 점수" 가 된다.
    # 첫 이벤트에서 NaN 이면 0 으로 채운다.
    df["current_home_points"] = (
        df.groupby("gameid", sort=False)["h_pts"].ffill().fillna(0.0).astype(np.float32)
    )
    df["current_away_points"] = (
        df.groupby("gameid", sort=False)["a_pts"].ffill().fillna(0.0).astype(np.float32)
    )

    # ----- team_side 계산 -----
    # 데이터셋에는 home/away 가 명시돼 있지 않다. 대신 누적 점수가 증가한 행의
    # team 코드가 곧 그 게임의 home(또는 away) 팀이라는 점을 이용해 추론한다.
    g = df.groupby("gameid", sort=False)
    prev_h = g["current_home_points"].shift(1).fillna(0.0)
    prev_a = g["current_away_points"].shift(1).fillna(0.0)
    home_scored = (df["current_home_points"] > prev_h) & df["team"].notna()
    away_scored = (df["current_away_points"] > prev_a) & df["team"].notna()

    # 노이즈 방지를 위해 가장 자주 등장한 team 코드를 그 사이드의 팀으로 채택
    home_by_game = (
        df.loc[home_scored]
        .groupby("gameid", sort=False)["team"]
        .agg(lambda s: s.mode().iloc[0])
    )
    away_by_game = (
        df.loc[away_scored]
        .groupby("gameid", sort=False)["team"]
        .agg(lambda s: s.mode().iloc[0])
    )

    # 모든 게임이 home/away 둘 다 식별되어야 함 (한쪽이라도 0점이면 추론 불가)
    all_games = set(df["gameid"].unique())
    missing_home = sorted(all_games - set(home_by_game.index))
    missing_away = sorted(all_games - set(away_by_game.index))
    if missing_home or missing_away:
        raise ValueError(
            f"Could not infer home/away for some games: "
            f"home_missing={len(missing_home)}, away_missing={len(missing_away)}"
        )

    # 행 단위 매핑 → team_side: 홈=1.0, 어웨이=0.0, team 없음(=무귀속)=0.5
    home_team_per_row = df["gameid"].map(home_by_game)
    away_team_per_row = df["gameid"].map(away_by_game)
    team_side = np.where(
        df["team"] == home_team_per_row, 1.0,
        np.where(df["team"] == away_team_per_row, 0.0, 0.5),
    )
    df["team_side"] = team_side.astype(np.float32)

    # 경기 경과 시간 계산
    #   - 정규 쿼터(1~4): 12분 = 720초
    #   - 연장(5 이상): 5분 = 300초
    period_len = np.where(df["period"].to_numpy() <= 4, 720.0, 300.0)
    elapsed_before = np.where(
        df["period"].to_numpy() <= 4,
        (df["period"].to_numpy() - 1) * 720.0,
        4 * 720.0 + (df["period"].to_numpy() - 5) * 300.0,
    )
    df["elapsed_game_sec"] = (elapsed_before + (period_len - df["clock_sec_remaining"])).astype(
        np.float32
    )

    # 게임별 최종 점수를 다시 join 해서 잔여 점수(타깃)를 계산
    final_scores = (
        df.groupby("gameid", sort=False)[["current_home_points", "current_away_points"]]
        .last()
        .rename(
            columns={
                "current_home_points": "final_home_points",
                "current_away_points": "final_away_points",
            }
        )
    )
    df = df.join(final_scores, on="gameid")

    df["remaining_home_points"] = (
        df["final_home_points"] - df["current_home_points"]
    ).astype(np.float32)
    df["remaining_away_points"] = (
        df["final_away_points"] - df["current_away_points"]
    ).astype(np.float32)

    # 이벤트 타입 → 정수 ID (LSTM 임베딩 입력용)
    df["event_type"] = [
        normalize_event_type(type_raw=t, desc=d)
        for t, d in zip(df["type"].values, df["desc"].values)
    ]

    event_vocab = sorted(df["event_type"].unique().tolist())
    event2id = {name: idx for idx, name in enumerate(event_vocab)}
    df["event_type_id"] = df["event_type"].map(event2id).astype(np.int64)

    # ----- 일관성 검증 -----

    # 모델에 들어갈 컬럼에 NaN 이 있으면 즉시 실패 (team_side 포함)
    if df[NUMERIC_FEATURE_COLUMNS + TARGET_COLUMNS + ["event_type_id"]].isna().any().any():
        raise ValueError("NaN detected after preprocessing")

    # team_side 는 {0.0, 0.5, 1.0} 셋 중 하나만 허용
    allowed_sides = {0.0, 0.5, 1.0}
    actual_sides = set(np.unique(df["team_side"].to_numpy()).tolist())
    if not actual_sides.issubset(allowed_sides):
        raise ValueError(f"Unexpected team_side values: {actual_sides - allowed_sides}")

    # 잔여 점수는 음수가 될 수 없다 (final >= current 이어야 함)
    if (df[TARGET_COLUMNS] < 0).any().any():
        raise ValueError("Remaining points must be non-negative")

    # 각 게임의 마지막 이벤트에서는 타깃이 (0, 0) 이어야 한다
    last_targets = df.groupby("gameid", sort=False)[TARGET_COLUMNS].last()
    if not (last_targets == 0).all().all():
        raise ValueError("Each game's final target must be (0, 0)")

    # period 는 게임 내에서 단조 증가
    period_diff = df.groupby("gameid", sort=False)["period"].diff()
    if (period_diff.dropna() < 0).any():
        raise ValueError("Period should be non-decreasing within each game")

    # 같은 period 내에서 clock_sec_remaining 은 단조 비증가
    prev_period = df.groupby("gameid", sort=False)["period"].shift(1)
    prev_clock = df.groupby("gameid", sort=False)["clock_sec_remaining"].shift(1)
    same_period_clock_violation = (df["period"] == prev_period) & (
        df["clock_sec_remaining"] > prev_clock
    )
    if same_period_clock_violation.any():
        raise ValueError("Clock should be non-increasing inside each period")

    return df


### DataFrame → 게임 단위 시퀀스 변환

게임마다 길이가 다르므로 한 게임을 한 샘플로 취급한다. 각 샘플은 `(seq_len,)` 형태의 이벤트 ID 텐서, `(seq_len, 6)` 수치 피처(시간/점수/`team_side` 포함), `(seq_len, 2)` 타깃을 포함하는 dict 다.

In [ ]:
def build_game_sequences(df: pd.DataFrame) -> list[dict]:
    """전처리된 DataFrame 을 게임 단위 텐서 시퀀스 리스트로 변환."""
    sequences: list[dict] = []

    for gameid, game_df in df.groupby("gameid", sort=False):
        event_type_id = torch.tensor(
            game_df["event_type_id"].to_numpy(),
            dtype=torch.long,
        )
        num_features = torch.tensor(
            game_df[NUMERIC_FEATURE_COLUMNS].to_numpy(dtype=np.float32),
            dtype=torch.float32,
        )
        targets = torch.tensor(
            game_df[TARGET_COLUMNS].to_numpy(dtype=np.float32),
            dtype=torch.float32,
        )

        sequences.append(
            {
                "gameid": str(gameid),
                "event_type_id": event_type_id,
                "num_features": num_features,
                "targets": targets,
                "seq_len": int(len(game_df)),
            }
        )

    return sequences


def split_by_gameid(
    sequences,
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
) -> Dict[str, list]:
    """gameid 기준으로 train/val/test 분할.

    같은 게임의 이벤트들이 서로 다른 split 에 들어가면 데이터 누수가 발생하므로,
    무작위 셔플 대신 gameid 를 정렬해 결정적으로 분리한다.
    """
    if train_ratio <= 0 or val_ratio <= 0 or train_ratio + val_ratio >= 1.0:
        raise ValueError("train_ratio and val_ratio must satisfy 0 < train, val and train+val < 1")

    seq_by_gid = {sample["gameid"]: sample for sample in sequences}
    # gameid 가 0으로 시작하는 문자열이라 정수 변환 후 정렬
    gameids_sorted = sorted(seq_by_gid.keys(), key=lambda x: int(x))

    n_games = len(gameids_sorted)
    n_train = int(n_games * train_ratio)
    n_val = int(n_games * val_ratio)

    train_ids = gameids_sorted[:n_train]
    val_ids = gameids_sorted[n_train : n_train + n_val]
    test_ids = gameids_sorted[n_train + n_val :]

    split = {
        "train": [seq_by_gid[g] for g in train_ids],
        "val": [seq_by_gid[g] for g in val_ids],
        "test": [seq_by_gid[g] for g in test_ids],
    }

    # split 간 gameid 가 겹치지 않는지 확인 (누수 방지)
    train_set, val_set, test_set = set(train_ids), set(val_ids), set(test_ids)
    if train_set & val_set or train_set & test_set or val_set & test_set:
        raise ValueError("Split leakage detected: overlapping gameid")

    return split


: 

### 전처리 실행 및 산출물 저장

`runs/train` 디렉터리가 이미 존재하면 `runs/train1`, `runs/train2`, ... 와 같이 숫자를 붙여 새 디렉터리를 만든다 (실행 결과를 덮어쓰지 않기 위함).

저장되는 파일:

- `train.pt` / `val.pt` / `test.pt` — 각 split 의 시퀀스 리스트
- `vocab_meta.pt` — event 사전, 피처 컬럼명, 타깃 컬럼명, split 별 게임 수

In [ ]:
# 전처리 → 시퀀스화 → split
processed_df = build_preprocessed_df(str(csv_path))
sequences = build_game_sequences(processed_df)
splits = split_by_gameid(sequences, train_ratio=0.8, val_ratio=0.1)

# 출력 디렉터리 자동 증분 (runs/train, runs/train1, runs/train2, ...)
base_path = "runs/train"
new_path = base_path
count = 1

while os.path.exists(new_path):
    new_path = f"{base_path}{count}"
    count += 1

out_dir = Path(new_path)
out_dir.mkdir(parents=True, exist_ok=True)

# 각 split 을 .pt 로 저장
for split_name in ["train", "val", "test"]:
    torch.save(splits[split_name], out_dir / f"{split_name}.pt")

# 이벤트 사전 (event_type ↔ id 매핑) 및 메타데이터 저장
vocab_table = (
    processed_df[["event_type", "event_type_id"]]
    .drop_duplicates()
    .sort_values("event_type_id")
)
event2id = {
    row.event_type: int(row.event_type_id)
    for row in vocab_table.itertuples(index=False)
}
id2event = {v: k for k, v in event2id.items()}

vocab_meta = {
    "event2id": event2id,
    "id2event": id2event,
    "numeric_feature_columns": NUMERIC_FEATURE_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "split_game_counts": {k: len(v) for k, v in splits.items()},
}
torch.save(vocab_meta, out_dir / "vocab_meta.pt")

print("Saved:", out_dir)
print("split game counts:", vocab_meta["split_game_counts"])
print("total events:", len(processed_df))
print("total games:", processed_df["gameid"].nunique())


: 

In [ ]:
# 저장된 .pt 파일을 다시 로드해 dtype, shape, split 누수 여부를 점검한다.
train_loaded = torch.load(out_dir / "train.pt")
val_loaded = torch.load(out_dir / "val.pt")
test_loaded = torch.load(out_dir / "test.pt")
meta_loaded = torch.load(out_dir / "vocab_meta.pt")

assert isinstance(train_loaded, list) and len(train_loaded) > 0
assert isinstance(val_loaded, list) and len(val_loaded) > 0
assert isinstance(test_loaded, list) and len(test_loaded) > 0

# 첫 샘플의 키, 텐서 dtype, shape 점검
sample = train_loaded[0]
assert set(sample.keys()) == {"gameid", "event_type_id", "num_features", "targets", "seq_len"}
assert sample["event_type_id"].dtype == torch.long
assert sample["num_features"].dtype == torch.float32
assert sample["targets"].dtype == torch.float32
assert sample["num_features"].shape[0] == sample["seq_len"]
assert sample["targets"].shape == (sample["seq_len"], 2)

# split 간 gameid 가 겹치지 않는지 다시 확인
train_ids = {x["gameid"] for x in train_loaded}
val_ids = {x["gameid"] for x in val_loaded}
test_ids = {x["gameid"] for x in test_loaded}
assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0

print("load/shape/split checks passed")
print("event vocab size:", len(meta_loaded["event2id"]))


: 

## Dataset / DataLoader

게임마다 길이가 다르므로 배치를 만들 때 **가장 긴 시퀀스 길이로 패딩**하고, 어느 위치가 실제 데이터인지 표시하는 `valid_mask` 를 함께 반환한다. 손실 계산 시 패딩 위치는 mask 로 무시된다.

In [ ]:
from pathlib import Path
from typing import Union

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

: 

In [ ]:
class GameSequenceDataset(Dataset):
    """게임 단위 시퀀스를 보관하는 Dataset.

    파일 경로 또는 이미 로드된 리스트 모두 받을 수 있다.
    """

    def __init__(self, data_source: Union[str, Path, list[dict]]):
        if isinstance(data_source, (str, Path)):
            samples = torch.load(Path(data_source))
        else:
            samples = list(data_source)

        if len(samples) == 0:
            raise ValueError("Dataset is empty")

        # 처음 5개 샘플의 키 형태가 기대 스키마와 같은지 점검
        required_keys = {"gameid", "event_type_id", "num_features", "targets", "seq_len"}
        for idx, sample in enumerate(samples[:5]):
            if set(sample.keys()) != required_keys:
                raise ValueError(f"Sample keys mismatch at index {idx}: {sample.keys()}")

        self.samples = samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> dict:
        return self.samples[index]

: 

In [ ]:
def collate_game_sequences(batch: list[dict], pad_event_id: int = 0) -> dict:
    """가변 길이 시퀀스를 배치 텐서로 묶는다.

    반환 dict:
        gameid:        list[str]  - 길이 B
        event_type_id: LongTensor - (B, T_max)
        num_features:  FloatTensor- (B, T_max, F)
        targets:       FloatTensor- (B, T_max, 2)
        lengths:       LongTensor - (B,)  실제 시퀀스 길이
        valid_mask:    BoolTensor - (B, T_max)  True 가 실제 데이터, False 가 패딩
    """
    gameids = [sample["gameid"] for sample in batch]
    event_ids = [sample["event_type_id"] for sample in batch]
    num_features = [sample["num_features"] for sample in batch]
    targets = [sample["targets"] for sample in batch]

    lengths = torch.tensor([sample["seq_len"] for sample in batch], dtype=torch.long)

    # 가장 긴 시퀀스 길이로 패딩
    event_ids_padded = pad_sequence(
        event_ids,
        batch_first=True,
        padding_value=pad_event_id,
    )
    num_features_padded = pad_sequence(
        num_features,
        batch_first=True,
        padding_value=0.0,
    )
    targets_padded = pad_sequence(
        targets,
        batch_first=True,
        padding_value=0.0,
    )

    # arange < lengths broadcasting → 각 행의 실제 길이만큼 True
    max_len = event_ids_padded.size(1)
    valid_mask = torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)

    return {
        "gameid": gameids,
        "event_type_id": event_ids_padded,
        "num_features": num_features_padded,
        "targets": targets_padded,
        "lengths": lengths,
        "valid_mask": valid_mask,
    }

: 

In [ ]:
def make_dataloaders(
    processed_dir: Union[str, Path],
    batch_size: int = 8,
    shuffle_train: bool = True,
    num_workers: int = 0,
) -> dict:
    """전처리된 디렉터리에서 train/val/test DataLoader 를 한 번에 생성."""
    processed_dir = Path(processed_dir)

    train_ds = GameSequenceDataset(processed_dir / "train.pt")
    val_ds = GameSequenceDataset(processed_dir / "val.pt")
    test_ds = GameSequenceDataset(processed_dir / "test.pt")

    loaders = {
        "train": DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=shuffle_train,
            num_workers=num_workers,
            collate_fn=collate_game_sequences,
        ),
        "val": DataLoader(
            val_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            collate_fn=collate_game_sequences,
        ),
        "test": DataLoader(
            test_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            collate_fn=collate_game_sequences,
        ),
    }
    return loaders


: 

In [ ]:
# 배치 한 개를 꺼내 shape 가 의도대로 나오는지 확인
loaders = make_dataloaders(processed_dir=out_dir, batch_size=4, shuffle_train=True)

first_batch = next(iter(loaders["train"]))
print("event_type_id:", first_batch["event_type_id"].shape)
print("num_features:", first_batch["num_features"].shape)
print("targets:", first_batch["targets"].shape)
print("valid_mask:", first_batch["valid_mask"].shape)
print("lengths:", first_batch["lengths"])


: 

## 모델 (Score Prediction LSTM)

구조:

```
event_type_id ─► nn.Embedding(num_event_types, event_emb_dim)
                                │
num_features ───────────────────┼─► concat ─► LSTM(hidden_size) ─► MLP ─► (remaining_home, remaining_away)
                                │
```

- 이벤트 타입은 임베딩(16차원), 수치 피처(6차원: 경과시간 / period / 남은시간 / 누적 홈·원정 점수 / `team_side`)와 concat 해 LSTM 입력으로 사용
- LSTM 은 `pack_padded_sequence` 로 패딩을 무시하면서 가변 길이 시퀀스를 처리
- 출력은 매 타임스텝에서 (잔여 홈 점수, 잔여 원정 점수) 두 값을 회귀

In [ ]:
import torch
import torch.nn as nn

: 

In [ ]:
class ScorePrediction(nn.Module):
    """이벤트 시퀀스로부터 매 시점의 잔여 점수(홈/원정)를 예측하는 LSTM."""

    def __init__(
        self,
        num_event_types: int,
        num_numeric_features: int,
        hidden_size: int = 128,
        num_layers: int = 2,
        event_emb_dim: int = 16,
        dropout: float = 0.2,
        output_size: int = 2,
    ):
        super().__init__()

        # 이벤트 타입(예: Made Shot, Rebound, ...) 임베딩
        self.event_emb = nn.Embedding(num_event_types, event_emb_dim)

        # 임베딩 + 수치 피처 → LSTM
        input_dim = event_emb_dim + num_numeric_features
        self.lstm = nn.LSTM(
            input_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # 매 타임스텝의 hidden state → (remaining_home, remaining_away)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, output_size),
        )

    def forward(self, event_type_ids, numeric_feats, lengths):
        from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

        # 이벤트 임베딩과 수치 피처를 마지막 차원에서 concat
        event_e = self.event_emb(event_type_ids)
        x = torch.cat([event_e, numeric_feats], dim=-1)

        # 가변 길이 → pack 으로 압축, LSTM 통과 후 다시 pad
        # enforce_sorted=False 로 두어 정렬 부담을 PyTorch 가 알아서 처리하게 함
        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=event_type_ids.size(1),
        )

        # 모든 타임스텝에 대해 회귀값 출력 → 손실 계산 시 valid_mask 로 패딩 제거
        pred = self.regressor(out)
        return pred


: 

## 학습 / 검증 루프

- 손실: SmoothL1 (Huber) — 큰 오차에 덜 민감해 회귀 안정성 확보
- 평가 지표: MAE (점수 단위 절대 오차)
- **모든 손실/지표는 `valid_mask` 로 패딩 위치를 제외**하고 계산
- LR 스케줄러: `ReduceLROnPlateau` (val loss 개선이 멈추면 lr × 0.5)
- best 체크포인트는 val MAE 기준으로 선택해 저장

In [ ]:
import copy
import time
import json

import torch.nn.functional as F

: 

In [ ]:
def masked_sums(preds, targets, valid_mask):
    """패딩 위치를 제외한 SmoothL1 합, L1 합, 유효 원소 수를 계산.

    valid_mask 는 (B, T) 라 마지막 타깃 차원(2)에 broadcast 하기 위해 unsqueeze 한다.
    """
    mask = valid_mask.unsqueeze(-1).float()
    smooth_l1_sum = (F.smooth_l1_loss(preds, targets, reduction="none") * mask).sum()
    l1_sum = ((preds - targets).abs() * mask).sum()
    # 유효 원소 수 = 유효 타임스텝 수 × 출력 차원(2)
    valid_values = int(valid_mask.sum().item()) * preds.size(-1)
    return smooth_l1_sum, l1_sum, max(valid_values, 1)

: 

In [ ]:
def run_one_epoch(model, loader, optimizer, device, grad_clip=1.0):
    """한 에폭 학습 또는 검증.

    optimizer 가 None 이면 검증 모드(역전파/업데이트 없음).
    """
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss_sum = 0.0
    total_l1_sum = 0.0
    total_values = 0

    for batch in loader:
        # 배치를 device 로 이동
        event_type_ids = batch["event_type_id"].to(device)
        numeric_feats = batch["num_features"].to(device)
        targets = batch["targets"].to(device)
        lengths = batch["lengths"].to(device)
        valid_mask = batch["valid_mask"].to(device)

        with torch.set_grad_enabled(is_train):
            preds = model(event_type_ids, numeric_feats, lengths)
            loss_sum, l1_sum, value_count = masked_sums(preds, targets, valid_mask)
            # 평균 손실로 백프롭 (배치 크기 효과 정규화)
            loss = loss_sum / value_count

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                # 그래디언트 폭발 방지
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

        total_loss_sum += float(loss_sum.detach().item())
        total_l1_sum += float(l1_sum.detach().item())
        total_values += value_count

    # 에폭 전체에 대한 가중 평균 (배치 길이가 달라도 공정한 평균)
    return {
        "loss": total_loss_sum / max(total_values, 1),
        "mae": total_l1_sum / max(total_values, 1),
    }

: 

In [ ]:
def fit(
    model,
    loaders,
    optimizer,
    scheduler,
    device,
    epochs,
    ckpt_path,
):
    """전체 학습 루프. val MAE 가 최저인 시점의 가중치를 ckpt_path 에 저장한다."""
    best_val_mae = float("inf")
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        # 학습 → 검증
        train_metrics = run_one_epoch(model, loaders["train"], optimizer, device)
        val_metrics = run_one_epoch(model, loaders["val"], optimizer=None, device=device)

        # val loss 가 정체되면 lr 감소
        scheduler.step(val_metrics["loss"])

        lr = optimizer.param_groups[0]["lr"]
        elapsed = time.time() - t0
        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_metrics['loss']:.4f}, train_mae={train_metrics['mae']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f}, val_mae={val_metrics['mae']:.4f} | "
            f"lr={lr:.2e} | {elapsed:.1f}s"
        )

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "train_mae": train_metrics["mae"],
                "val_loss": val_metrics["loss"],
                "val_mae": val_metrics["mae"],
                "lr": lr,
            }
        )

        # 최저 val MAE 가중치 보관 (epoch 끝까지 학습 후 한 번에 저장)
        if val_metrics["mae"] < best_val_mae:
            best_val_mae = val_metrics["mae"]
            best_state = copy.deepcopy(model.state_dict())

    # 학습 종료 후 best 상태로 복원하여 디스크에 기록
    if best_state is not None:
        model.load_state_dict(best_state)
        torch.save(
            {
                "model_state_dict": best_state,
                "best_val_mae": best_val_mae,
            },
            ckpt_path,
        )
        print(f"Best checkpoint saved to: {ckpt_path}")

    return history

: 

In [ ]:
# 재현성을 위한 시드 고정
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# CUDA 가능하면 GPU, 아니면 CPU 사용
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# DataLoader 재생성 (학습용 batch_size 로)
batch_size = 8
loaders = make_dataloaders(processed_dir=out_dir, batch_size=batch_size, shuffle_train=True)

# 모델 하이퍼파라미터
num_event_types = len(meta_loaded["event2id"])
num_numeric_features = len(meta_loaded["numeric_feature_columns"])

model = ScorePrediction(
    num_event_types=num_event_types,
    num_numeric_features=num_numeric_features,
    hidden_size=128,
    num_layers=2,
    event_emb_dim=16,
    dropout=0.2,
    output_size=2,
).to(device)

# AdamW + weight decay 로 약한 정규화
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
# val loss 가 2 epoch 동안 개선 없으면 lr 을 절반으로 줄인다
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

# 학습 시작
history = fit(
    model=model,
    loaders=loaders,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=10,
    ckpt_path=str(out_dir / "best_score_prediction.pt"),
)

# 최종 평가 (best 가중치가 model 에 로드된 상태)
val_metrics = run_one_epoch(model, loaders["val"], optimizer=None, device=device)
test_metrics = run_one_epoch(model, loaders["test"], optimizer=None, device=device)

print("Final val:", val_metrics)
print("Final test:", test_metrics)

# 학습 history 를 JSON 으로 저장 (이후 시각화에 사용)
history_path = out_dir / "history.json"
history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

print("History saved to:", history_path)

: 

## 학습 곡선 시각화 (Loss / MAE)

`history.json` 을 읽어 train/val Loss 와 MAE 를 epoch 별로 그린다. 결과 PNG 는 `runs/trainN/train_val_curves.png` 에 저장된다.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

# 학습 단계에서 저장한 history.json 을 다시 로드
history_path = out_dir / "history.json"
if history_path.exists():
    history_df = pd.DataFrame(json.loads(history_path.read_text(encoding="utf-8")))
else:
    raise FileNotFoundError("training history not found. Run training cell first.")

# 좌: SmoothL1 Loss / 우: MAE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("SmoothL1")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_mae"], marker="o", label="train")
axes[1].plot(history_df["epoch"], history_df["val_mae"], marker="o", label="val")
axes[1].set_title("MAE")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE (points)")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
curve_path = out_dir / "train_val_curves.png"
plt.savefig(curve_path, dpi=150)
plt.show()
print(f"Saved: {curve_path}")


: 

## 테스트셋 예측 시각화

베스트 체크포인트로 테스트셋 추론을 다시 수행하고:

1. 샘플 게임 3개의 시간축 곡선 — 각 게임마다 4개 패널을 그린다
   - 잔여 점수 (홈/원정): 실제(true) vs 예측(pred)
   - **최종 점수** (홈/원정): 실제 최종 점수 vs **현재 점수 + 예측 잔여** (pred_final). 이벤트가 진행될수록 예측 최종이 실제 최종으로 수렴해야 함
2. 전체 산점도 — 모든 (true, pred) 쌍을 뿌려 대각선과의 편차를 한눈에 확인
3. 홈/원정 각각의 평균 절대 오차(MAE) 출력

산출물은 `runs/trainN/` 아래 `test_predictions.csv`, `test_prediction_timeline.png`, `test_prediction_scatter.png` 로 저장된다.

In [ ]:
import numpy as np


# numeric_feats 에서 누적 점수 컬럼이 몇 번째 인덱스인지 메타로부터 조회
num_cols = meta_loaded["numeric_feature_columns"]
HOME_PTS_IDX = num_cols.index("current_home_points")
AWAY_PTS_IDX = num_cols.index("current_away_points")


def collect_test_predictions(model, loader, device):
    """테스트 로더 전체에 대해 예측을 모아 long-format DataFrame 으로 반환.

    valid_mask 로 패딩 위치를 잘라내어 (gameid, step) 단위로 한 행씩 만든다.
    최종 점수 시각화에 쓰기 위해 누적 점수도 함께 기록한다.
    """
    model.eval()
    rows = []

    with torch.no_grad():
        for batch in loader:
            event_type_ids = batch["event_type_id"].to(device)
            numeric_feats = batch["num_features"].to(device)
            targets = batch["targets"].to(device)
            lengths = batch["lengths"].to(device)
            valid_mask = batch["valid_mask"].to(device)
            gameids = batch["gameid"]

            preds = model(event_type_ids, numeric_feats, lengths)

            # 배치 내 각 게임에 대해 패딩을 제외하고 step 별 예측/정답을 기록
            for i, gid in enumerate(gameids):
                valid_idx = valid_mask[i].cpu().numpy().astype(bool)
                pred_np = preds[i].detach().cpu().numpy()[valid_idx]
                tgt_np = targets[i].detach().cpu().numpy()[valid_idx]
                num_np = numeric_feats[i].detach().cpu().numpy()[valid_idx]
                cur_h = num_np[:, HOME_PTS_IDX]
                cur_a = num_np[:, AWAY_PTS_IDX]

                for step, (p, y, ch, ca) in enumerate(zip(pred_np, tgt_np, cur_h, cur_a)):
                    rows.append(
                        {
                            "gameid": gid,
                            "step": int(step),
                            "current_home_points": float(ch),
                            "current_away_points": float(ca),
                            "pred_home_remaining": float(p[0]),
                            "pred_away_remaining": float(p[1]),
                            "true_home_remaining": float(y[0]),
                            "true_away_remaining": float(y[1]),
                        }
                    )

    return pd.DataFrame(rows)


# 베스트 체크포인트 로드 (현재 model 인스턴스에 best 가중치 덮어쓰기)
ckpt_path = out_dir / "best_score_prediction.pt"
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    if "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])

pred_df = collect_test_predictions(model, loaders["test"], device)
if pred_df.empty:
    raise ValueError("No predictions collected from test loader")

# 최종 점수 계산: 현재 누적 점수 + 잔여 점수
#   - true_*_final 은 게임 내내 상수 (실제 최종 점수)
#   - pred_*_final 은 이벤트가 진행될수록 true_*_final 로 수렴해야 함
pred_df["pred_home_final"] = pred_df["current_home_points"] + pred_df["pred_home_remaining"]
pred_df["pred_away_final"] = pred_df["current_away_points"] + pred_df["pred_away_remaining"]
pred_df["true_home_final"] = pred_df["current_home_points"] + pred_df["true_home_remaining"]
pred_df["true_away_final"] = pred_df["current_away_points"] + pred_df["true_away_remaining"]

# 분석/리포팅용 CSV 저장 (현재 점수 / 최종 점수 컬럼 포함)
pred_df.to_csv(out_dir / "test_predictions.csv", index=False)

# 1) 샘플 게임 3개의 시간축 곡선
#    각 게임마다 4개 패널: 홈 잔여, 어웨이 잔여, 홈 최종, 어웨이 최종
sample_gameids = pred_df["gameid"].drop_duplicates().head(3).tolist()
fig, axes = plt.subplots(
    len(sample_gameids), 4,
    figsize=(20, 3.2 * len(sample_gameids)),
    sharex=False,
)
if len(sample_gameids) == 1:
    axes = np.array([axes])

for r, gid in enumerate(sample_gameids):
    g = pred_df[pred_df["gameid"] == gid].sort_values("step")

    # 홈 잔여 점수
    axes[r, 0].plot(g["step"], g["true_home_remaining"], label="true", linewidth=2)
    axes[r, 0].plot(g["step"], g["pred_home_remaining"], label="pred", alpha=0.8)
    axes[r, 0].set_title(f"Game {gid} - Home Remaining")
    axes[r, 0].set_xlabel("Event step")
    axes[r, 0].set_ylabel("Points")
    axes[r, 0].grid(alpha=0.3)
    axes[r, 0].legend()

    # 어웨이 잔여 점수
    axes[r, 1].plot(g["step"], g["true_away_remaining"], label="true", linewidth=2)
    axes[r, 1].plot(g["step"], g["pred_away_remaining"], label="pred", alpha=0.8)
    axes[r, 1].set_title(f"Game {gid} - Away Remaining")
    axes[r, 1].set_xlabel("Event step")
    axes[r, 1].set_ylabel("Points")
    axes[r, 1].grid(alpha=0.3)
    axes[r, 1].legend()

    # 홈 최종 점수: true_final 은 상수 가로선, pred_final = current + pred_remaining
    axes[r, 2].plot(g["step"], g["true_home_final"], label="true_final", linewidth=2)
    axes[r, 2].plot(g["step"], g["pred_home_final"], label="pred_final", alpha=0.8)
    axes[r, 2].set_title(f"Game {gid} - Home Final")
    axes[r, 2].set_xlabel("Event step")
    axes[r, 2].set_ylabel("Points")
    axes[r, 2].grid(alpha=0.3)
    axes[r, 2].legend()

    # 어웨이 최종 점수
    axes[r, 3].plot(g["step"], g["true_away_final"], label="true_final", linewidth=2)
    axes[r, 3].plot(g["step"], g["pred_away_final"], label="pred_final", alpha=0.8)
    axes[r, 3].set_title(f"Game {gid} - Away Final")
    axes[r, 3].set_xlabel("Event step")
    axes[r, 3].set_ylabel("Points")
    axes[r, 3].grid(alpha=0.3)
    axes[r, 3].legend()

plt.tight_layout()
timeline_path = out_dir / "test_prediction_timeline.png"
plt.savefig(timeline_path, dpi=150)
plt.show()
print(f"Saved: {timeline_path}")

# 2) 전체 산점도 — 점이 빨간 대각선 위에 있을수록 예측이 정확하다
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

max_home = max(pred_df["true_home_remaining"].max(), pred_df["pred_home_remaining"].max())
max_away = max(pred_df["true_away_remaining"].max(), pred_df["pred_away_remaining"].max())

axes[0].scatter(pred_df["true_home_remaining"], pred_df["pred_home_remaining"], s=8, alpha=0.25)
axes[0].plot([0, max_home], [0, max_home], "r--", linewidth=1)
axes[0].set_title("Home: True vs Pred")
axes[0].set_xlabel("True")
axes[0].set_ylabel("Pred")
axes[0].grid(alpha=0.3)

axes[1].scatter(pred_df["true_away_remaining"], pred_df["pred_away_remaining"], s=8, alpha=0.25)
axes[1].plot([0, max_away], [0, max_away], "r--", linewidth=1)
axes[1].set_title("Away: True vs Pred")
axes[1].set_xlabel("True")
axes[1].set_ylabel("Pred")
axes[1].grid(alpha=0.3)

plt.tight_layout()
scatter_path = out_dir / "test_prediction_scatter.png"
plt.savefig(scatter_path, dpi=150)
plt.show()
print(f"Saved: {scatter_path}")

# 3) 홈/원정 각각의 MAE 와 전체 평균
mae_home = (pred_df["pred_home_remaining"] - pred_df["true_home_remaining"]).abs().mean()
mae_away = (pred_df["pred_away_remaining"] - pred_df["true_away_remaining"]).abs().mean()
print(f"Test MAE - home: {mae_home:.4f}, away: {mae_away:.4f}, overall: {(mae_home + mae_away) / 2:.4f}")
